In [10]:
import pandas as pd
import os
import toml

In [11]:
config = toml.load("config.toml")

In [12]:
# Load network as shapefile and intersect with Census tracts
import geopandas as gpd
gdf_network = gpd.read_file(os.path.join(config["model_run_dir_2023"], "inputs", "scenario", "networks", "shapefiles", "AM", "AM_edges.shp"))

In [13]:
import psrcelmerpy
import toml
config = toml.load("config.toml")

eg_conn = psrcelmerpy.ElmerGeoConn()
tract_gdf = eg_conn.read_geolayer('tract2020')
tract_gdf.to_crs(epsg=config["project_epsg"], inplace=True)

In [ ]:
# Split network links at tract boundaries, then assign geoid20 and segment length
if gdf_network.crs != tract_gdf.crs:
    gdf_network = gdf_network.to_crs(tract_gdf.crs)

network_with_tracts = gpd.overlay(
    gdf_network[['link_id', 'geometry']],
    tract_gdf[['geoid20', 'geometry']],
    how='intersection',
    keep_geom_type=True,
    make_valid=True,
)

# Length in miles for each tract-segmented link piece
network_with_tracts['length'] = network_with_tracts.geometry.length / 5280.0

In [17]:
# Load network results and merge with network_with_tracts to calculate total VMT within each tract
df_network = pd.read_csv(os.path.join(config["model_run_dir_2023"], "outputs", "network", "network_results.csv"))

In [18]:
gdf_network.columns

Index(['direction', 'i', 'j', 'length', 'modes', 'type', 'lanes', 'vdf', 'ul1',
       'ul2', 'ul3', 'toll1', 'toll2', 'toll3', 'trkc1', 'trkc2', 'trkc3',
       'ttf', 'PSRCEdgeID', 'FacilityTy', 'Processing', 'projRteID', 'CountID',
       'CountyID', 'CorridorID', 'FGTS', 'is_managed', 'bkfac', 'link_id',
       'weight', 'upslp', 'geometry'],
      dtype='object')

In [ ]:
df = df_network.merge(
    network_with_tracts[['link_id', 'geoid20', 'length']],
    left_on='ij',
    right_on='link_id',
    how='left',
    suffixes=('', '_segment'),
)

# Use split segment length for tract-level VMT; keep original when no tract match
df['length'] = df['length_segment'].fillna(df['length'])
df = df.drop(columns=['length_segment'])

In [20]:
# Calculate VMT from volume
# Remove links with facility type = 0 from the calculation
df["facility_type"] = df["data3"].copy()  # Rename for human readability
df = df[df["facility_type"] > 0]

# Calculate VMT by bus, SOV, HOV2, HOV3+, medium truck, heavy truck
df["sov_vol"] = df["@sov_inc1"] + df["@sov_inc2"] + df["@sov_inc3"]
df["sov_vmt"] = df["sov_vol"] * df["length"]
df["hov2_vol"] = df["@hov2_inc1"] + df["@hov2_inc2"] + df["@hov2_inc3"]
df["hov2_vmt"] = df["hov2_vol"] * df["length"]
df["hov3_vol"] = df["@hov3_inc1"] + df["@hov3_inc2"] + df["@hov3_inc3"]
df["hov3_vmt"] = df["hov3_vol"] * df["length"]
df["tnc_vmt"] = df["@tnc_inc1"] + df["@tnc_inc2"] + df["@tnc_inc3"]
df["medium_truck_vmt"] = df["@mveh"] * df["length"]
df["heavy_truck_vmt"] = df["@hveh"] * df["length"]

In [21]:
df.groupby("geoid20")[["sov_vmt", "hov2_vmt", "hov3_vmt", "tnc_vmt", "medium_truck_vmt", "heavy_truck_vmt"]].sum()

,sov_vmt,hov2_vmt,hov3_vmt,tnc_vmt,medium_truck_vmt,heavy_truck_vmt
geoid20,,,,,,
53033000101,20920.720656,4879.301301,1965.528248,695.206299,609.616462,19.911482
53033000102,24519.462194,5798.175032,2330.086479,1167.008545,688.740066,21.323915
53033000201,37098.482377,9615.531126,3963.175600,2047.015137,1601.781592,111.245462
53033000202,14586.957243,4014.680164,1583.904586,704.706543,440.695716,39.747291
53033000300,215220.498440,40783.437313,18325.298266,3527.528321,17240.516025,12294.129165
...,...,...,...,...,...,...
53061053802,131884.953901,34809.425747,15999.626627,221.205566,4993.871175,14166.295080
53061053803,66162.341970,16178.455716,7491.354742,112.103760,1671.546842,13688.204219
53061940001,551066.758515,93247.103148,44896.155575,1668.714844,20773.528679,55641.881078


In [22]:
df[["sov_vmt", "hov2_vmt", "hov3_vmt", "tnc_vmt", "medium_truck_vmt", "heavy_truck_vmt"]].to_csv(os.path.join(config["working_dir"], "vmt_by_tract.csv"), index=False)